# ATLAS Wind IDW Interpolation Tutorial

This notebook interpolates station based wind observations onto the high resolution grid produced by the wind downscaling workflow.

The workflow is organised as follows:
1. set the input parameters;
2. load the downscaled grid coordinates;
3. load wind speed and wind direction station observations;
4. convert speed and direction into wind components `u` and `v`;
5. apply Inverse Distance Weighting, IDW;
6. save one NetCDF file per month and per wind component.

All paths are relative and anonymous. Each team only needs to adapt the country name, the area name and the input file names if their local folder structure is different.

## Methodology Description:
This notebook uses the **Inverse Distance Weighting (IDW)** interpolation method to transform point observations from meteorological stations into a continuous high-resolution raster surface.

The input dataset consists of **scattered station measurements**, where each station provides the observed value of solar radiation at a specific location. The objective is to estimate solar radiation at the unknown cells of the final **90 m resolution grid**.

For each grid cell, the interpolated value is computed as a weighted average of the surrounding stations. The weights are assigned according to the inverse of the distance between the grid cell and each station, giving greater importance to nearby observations. In the current implementation, the weight is proportional to the inverse cube of the distance:

$$
w_i = \frac{1}{d_i^3}
$$

where:

* $w_i$ is the weight assigned to station *i*
* $d_i$ is the distance between the interpolation point and station *i*

The interpolated value is then calculated as:

$$
\hat{x} = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}
$$

where:

* $\hat{x}$ is the predicted value at the target grid cell
* $x_i$ is the observed value at station *i*
* $w_i$ is the corresponding distance-based weight

As a result, stations located closer to the target location have a much stronger influence on the predicted value than more distant stations. This approach allows the generation of a continuous 90 m resolution raster while preserving the spatial patterns represented by the station observations and ensuring that local measurements have the greatest influence on nearby areas.

## Step 1. Input parameters

Edit only this cell before running the notebook.

Expected folders:

`../data/stations/{country}/`

contains the station CSV files for wind speed and wind direction.

`../data/downscaled_data/{country}/sub_areas/`

contains the high resolution wind downscaling output used only to read the final grid coordinates.

`../data/stations/{country}/idw/`

will contain the IDW output NetCDF files created by this notebook.

In [1]:
from pathlib import Path

# Country name used in folder and file names.
# Example values: "argentina", "chile", "peru".
country = "chile"

# Area name used in the downscaling file name.
# Use the same value used in the wind downscaling notebook.
# Typical values are "continental" or "islands".
area_name = "continental"

# Optional bounding box used to keep only the stations inside the target area.
# Format: [lat_max, lon_min, lat_min, lon_max]
# Set area_bbox = None to use all stations available in the station CSV files.

# Example for an island area:
#area_bbox = [-25.0, -109.8, -34.8, -77.8]
# For continental Chile:
area_bbox = [-17.3,  -76.2,  -56.7,  -66.2]

# Months to process.
# Use one month, for example [1], or all months with list(range(1, 13)).
months_to_process = list(range(1, 2))
month = 1
# Names of the variables in the station CSV files.
# Change these only if the station files use different column names.
wind_speed_column = "wind_speed"
wind_direction_column = "wind_direction"

# IDW settings.
# power controls how fast the station influence decreases with distance.
# k is the number of nearest stations used for each grid point.
# chunk_size controls memory use for large grids.
idw_power = 3
idw_k = 12
chunk_size = 100_000

# Input folder containing station observations.
stations_path = Path(f"../data/stations/{country}/")

# CSV files containing monthly or time series station data.
wind_speed_file = stations_path / f"allstats_wind_speed_{country}.csv"
wind_direction_file = stations_path / f"allstats_wind_direction_{country}.csv"

# Folder containing the downscaled wind files from the previous workflow.
coords_path = Path(f"../data/downscaled_data/{country}/sub_areas/")

# Template of the downscaled files used to read the target grid coordinates.
# The notebook reads latitude and longitude from the first existing file matching this template.
coords_file_template = f"v10m_component_downscaled_{country}_m{month}_{area_name}.nc"

# Output folder for IDW products.
output_path = Path(f"../data/stations/{country}/idw/")
output_path.mkdir(parents=True, exist_ok=True)

## Step 2. Import libraries

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

from scipy.spatial import cKDTree

## Step 3. Helper functions

These functions read the grid, read station observations, prepare monthly data, convert wind speed and direction into components, apply IDW and save the results.

In [3]:
def get_existing_coords_file(coords_folder, template, months, country_name, area_name_value):
    """Return the first available downscaled file used to read the target grid."""
    for month in months:
        candidate = coords_folder / template.format(
            country=country_name,
            month=month,
            area_name=area_name_value,
        )
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"No coordinate file found in {coords_folder}. "
        f"Expected files following this template: {template}"
    )


def read_target_grid(coords_file):
    """Read latitude and longitude from a downscaled NetCDF file."""
    ds = xr.open_dataset(coords_file)

    if "longitude" not in ds or "latitude" not in ds:
        raise KeyError("The coordinate file must contain 'longitude' and 'latitude' variables.")

    lon = ds["longitude"].values
    lat = ds["latitude"].values

    lon = lon[~np.isnan(lon)]
    lat = lat[~np.isnan(lat)]

    # Use unique sorted coordinates to create a clean regular grid.
    # Latitude is sorted from north to south so maps start from the upper left corner.
    lat = np.sort(np.unique(lat))[::-1]
    lon = np.sort(np.unique(lon))

    ds.close()
    return lat, lon


def read_station_dataset(filepath, required_value_column):
    """Read a station CSV file and check the minimum required columns."""
    if not Path(filepath).exists():
        raise FileNotFoundError(f"Station file not found: {filepath}")

    df = pd.read_csv(filepath)

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
    elif "month" not in df.columns:
        raise ValueError("The station file must contain either a 'time' column or a 'month' column.")

    required_columns = {"name", "latitude", "longitude", required_value_column}
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {filepath}: {sorted(missing)}")

    return df


def choose_merge_keys(speed_df, direction_df):
    """Choose safe columns for merging speed and direction observations."""
    possible_keys = ["time", "month", "id", "name", "latitude", "longitude", "altitude"]
    keys = [col for col in possible_keys if col in speed_df.columns and col in direction_df.columns]

    if not keys:
        raise ValueError("No common columns found to merge wind speed and wind direction files.")

    # At least one temporal key is strongly recommended.
    if "time" not in keys and "month" not in keys:
        raise ValueError("The two wind files should share either a 'time' column or a 'month' column.")

    return keys


def merge_speed_and_direction(speed_df, direction_df):
    """Merge wind speed and direction data into a single station dataframe."""
    merge_keys = choose_merge_keys(speed_df, direction_df)

    direction_columns = merge_keys + [wind_direction_column]
    merged = speed_df.merge(
        direction_df[direction_columns],
        on=merge_keys,
        how="inner",
    )

    if merged.empty:
        raise ValueError("The merge between wind speed and wind direction produced no records.")

    return merged


def add_wind_components(df):
    """Convert wind speed and meteorological wind direction into u and v components.

    The meteorological convention expresses the direction from which the wind blows.
    With this convention:
    u = negative speed times sin(direction)
    v = negative speed times cos(direction)
    """
    out = df.copy()
    theta = np.deg2rad(out[wind_direction_column])

    out["u"] = -out[wind_speed_column] * np.sin(theta)
    out["v"] = -out[wind_speed_column] * np.cos(theta)

    return out


def prepare_station_monthly_data(group):
    """Convert station observations to monthly values.

    If the input contains a time column, the data are first aggregated to hourly values,
    then to daily values and finally to monthly means.

    If the input already contains a month column, the data are used as monthly values.
    """
    if "time" in group.columns:
        hourly = group.resample("60min", on="time").mean(numeric_only=True)
        daily = hourly.reset_index().resample("D", on="time").mean(numeric_only=True)
        monthly = daily.reset_index().resample("M", on="time").mean(numeric_only=True)
        monthly = monthly.groupby(monthly.index.month).mean(numeric_only=True)
    elif "month" in group.columns:
        columns_to_drop = [col for col in ["Unnamed: 0"] if col in group.columns]
        monthly = group.drop(columns=columns_to_drop).set_index("month")
    else:
        raise ValueError("The station data must contain either 'time' or 'month'.")

    return monthly


def filter_stations_by_bbox(station_monthly_df, bbox):
    """Keep only stations inside the selected bounding box."""
    if bbox is None:
        return station_monthly_df

    lat_max, lon_min, lat_min, lon_max = bbox

    filtered = station_monthly_df[
        (station_monthly_df["longitude"] >= lon_min)
        & (station_monthly_df["longitude"] <= lon_max)
        & (station_monthly_df["latitude"] >= lat_min)
        & (station_monthly_df["latitude"] <= lat_max)
    ]

    if filtered.empty:
        raise ValueError("No stations found inside the selected area_bbox.")

    return filtered


def simple_idw_chunked(pos, values, target_pos, power=3, k=12, chunk_size=100_000):
    """Apply IDW using nearest neighbours and chunks to reduce memory use."""
    if len(pos) == 0:
        raise ValueError("No station points available for IDW.")

    tree = cKDTree(pos)
    out = np.empty(target_pos.shape[0], dtype=np.float32)

    neighbours = min(k, len(pos))

    for start in range(0, target_pos.shape[0], chunk_size):
        end = min(start + chunk_size, target_pos.shape[0])
        points = target_pos[start:end]

        dist, idx = tree.query(points, k=neighbours)

        # When there is only one neighbour, scipy returns one dimensional arrays.
        if neighbours == 1:
            dist = dist[:, None]
            idx = idx[:, None]

        dist = np.maximum(dist, 1e-12)
        weights = 1.0 / dist**power
        weights = weights / weights.sum(axis=1, keepdims=True)

        out[start:end] = np.sum(weights * values[idx], axis=1)

    return out


def apply_idw(station_month_df, lat_new, lon_new, component, power=3, k=12, chunk_size=100_000):
    """Interpolate one wind component to the target grid using IDW."""
    month_df = station_month_df.dropna(subset=["latitude", "longitude", component])

    if month_df.empty:
        raise ValueError(f"No valid station data available for component {component}.")

    lat = month_df["latitude"].values.astype(np.float32)
    lon = month_df["longitude"].values.astype(np.float32)
    values = month_df[component].values.astype(np.float32)

    station_positions = np.column_stack([lat, lon])

    lat2d, lon2d = np.meshgrid(lat_new, lon_new, indexing="ij")
    target_grid = np.column_stack([lat2d.ravel(), lon2d.ravel()]).astype(np.float32)

    interpolated = simple_idw_chunked(
        station_positions,
        values,
        target_grid,
        power=power,
        k=k,
        chunk_size=chunk_size,
    )

    return interpolated.reshape(len(lat_new), len(lon_new))


def save_netcdf(ds, output_file):
    """Save a dataset as a compressed NetCDF file."""
    encoding = {var: {"zlib": True, "complevel": 1} for var in ds.data_vars}
    ds.to_netcdf(output_file, engine="netcdf4", encoding=encoding)


def run_idw_for_month_and_component(
    station_monthly_df,
    month,
    lat_grid,
    lon_grid,
    component,
    country_name,
    area_name_value,
    out_folder,
):
    """Run IDW for one month and one wind component, then save the result."""
    month_df = station_monthly_df[station_monthly_df.index == month]

    if month_df.empty:
        raise ValueError(f"No station data available for month {month}.")

    idw_array = apply_idw(
        month_df,
        lat_grid,
        lon_grid,
        component,
        power=idw_power,
        k=idw_k,
        chunk_size=chunk_size,
    )

    ds = xr.DataArray(
        idw_array,
        coords=[("latitude", lat_grid), ("longitude", lon_grid)],
        name=f"{component}_reshaped",
        attrs={"units": "m s-1"},
    ).to_dataset()

    output_file = out_folder / f"{component}_idw_{country_name}_m{month}_{area_name_value}.nc"
    save_netcdf(ds, output_file)

    return output_file


## Step 4. Load the target grid and station observations

This cell reads the latitude and longitude of the high resolution grid, then reads the station files and creates the wind components `u` and `v`.

In [4]:
coords_file = get_existing_coords_file(
    coords_path,
    coords_file_template,
    months_to_process,
    country,
    area_name,
)

lat_grid, lon_grid = read_target_grid(coords_file)

wind_speed_df = read_station_dataset(wind_speed_file, wind_speed_column)
wind_direction_df = read_station_dataset(wind_direction_file, wind_direction_column)

wind_df = merge_speed_and_direction(wind_speed_df, wind_direction_df)
wind_df = add_wind_components(wind_df)

station_monthly_df = wind_df.groupby("name", group_keys=False).apply(prepare_station_monthly_data)
station_monthly_df = filter_stations_by_bbox(station_monthly_df, area_bbox)

print(f"Coordinate file used: {coords_file}")
print(f"Number of monthly station records available: {len(station_monthly_df)}")
print(f"Number of stations available: {station_monthly_df['name'].nunique() if 'name' in station_monthly_df.columns else 'not available'}")

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Coordinate file used: ../data/downscaled_data/chile/sub_areas/v10m_component_downscaled_chile_m1_continental.nc
Number of monthly station records available: 1368
Number of stations available: 114


## Step 5. Run IDW interpolation

This cell creates one NetCDF file for `u` and one NetCDF file for `v` for each selected month.

The files are saved in:

`../data/stations/{country}/idw/`

In [5]:
created_files = []

for month in months_to_process:
    for component in ["u", "v"]:
        output_file = run_idw_for_month_and_component(
            station_monthly_df=station_monthly_df,
            month=month,
            lat_grid=lat_grid,
            lon_grid=lon_grid,
            component=component,
            country_name=country,
            area_name_value=area_name,
            out_folder=output_path,
        )
        created_files.append(output_file)
        print(f"Created: {output_file}")

created_files

Created: ../data/stations/chile/idw/u_idw_chile_m1_continental.nc
Created: ../data/stations/chile/idw/v_idw_chile_m1_continental.nc


[PosixPath('../data/stations/chile/idw/u_idw_chile_m1_continental.nc'),
 PosixPath('../data/stations/chile/idw/v_idw_chile_m1_continental.nc')]

## Step 6. Optional quick check

Run this cell only if you want to open one output file and check that the grid was saved correctly.

In [6]:
if created_files:
    example_ds = xr.open_dataset(created_files[0])
    display(example_ds)
else:
    print("No files were created.")

<xarray.Dataset> Size: 2GB
Dimensions:     (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude    (latitude) float32 187kB -17.5 -17.5 -17.5 ... -56.54 -56.54
  * longitude   (longitude) float32 42kB -75.72 -75.72 -75.72 ... -66.93 -66.93
Data variables:
    u_reshaped  (latitude, longitude) float32 2GB ...

## Notes for country teams

Before running the notebook, check that the station files contain the columns `name`, `latitude`, `longitude` and either `time` or `month`.

The wind speed file must contain the column set in `wind_speed_column`.

The wind direction file must contain the column set in `wind_direction_column`.

Wind direction must follow the meteorological convention, meaning the direction from which the wind blows.

The coordinate file must contain `latitude` and `longitude` variables.

If the notebook is slow for very large areas, reduce `chunk_size` or `idw_k`.